# 39. FastAPI — `fastapi`
**Python Libraries Lab · Phase 3 · Web & Networking**

_Modern, fast APIs with automatic validation and docs._

⚙️ **Runs in Colab** after the setup cell below.

FastAPI is a modern framework for building APIs. You describe the data your API accepts using Python type hints and Pydantic models, and FastAPI does the boring parts for you. It checks incoming data, returns clear errors, converts to and from JSON, and builds interactive documentation where anyone can try your API in the browser.

**In this notebook:** You will build the ShopEasy products and orders API with models, validation, query filters, error handling and a simple API key check. You will test it with FastAPI's test client, then run it with Uvicorn and explore the automatic docs in Colab.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q "fastapi[standard]"
import importlib.metadata as md
for pkg in ['fastapi']:
    print(pkg, md.version(pkg))

## Step 1 · Write the API
Saved as `main.py` so Uvicorn can run it. Read the type hints: they drive validation and docs.

In [ ]:
%%writefile main.py
from fastapi import FastAPI, HTTPException, Header, Depends, Query
from pydantic import BaseModel, Field

app = FastAPI(title="ShopEasy API", version="1.0")

class ProductIn(BaseModel):
    name: str = Field(min_length=2, max_length=60)
    price: float = Field(gt=0, description="Price in rupees")
    in_stock: bool = True

class Product(ProductIn):
    id: int

DB: dict[int, Product] = {
    1: Product(id=1, name="Steel Bottle", price=499),
    2: Product(id=2, name="Yoga Mat", price=1299),
    3: Product(id=3, name="Desk Lamp", price=899, in_stock=False),
}

def require_key(x_api_key: str = Header(default="")):
    if x_api_key != "demo-key":
        raise HTTPException(status_code=401, detail="Send header X-API-Key: demo-key")

@app.get("/products", response_model=list[Product])
def list_products(max_price: float | None = Query(default=None, gt=0), in_stock: bool | None = None):
    items = DB.values()
    if max_price is not None:
        items = [p for p in items if p.price <= max_price]
    if in_stock is not None:
        items = [p for p in items if p.in_stock == in_stock]
    return list(items)

@app.get("/products/{pid}", response_model=Product)
def get_product(pid: int):
    if pid not in DB:
        raise HTTPException(status_code=404, detail="Product not found")
    return DB[pid]

@app.post("/products", response_model=Product, status_code=201, dependencies=[Depends(require_key)])
def create_product(item: ProductIn):
    new = Product(id=max(DB) + 1, **item.model_dump())
    DB[new.id] = new
    return new

## Step 2 · Test with TestClient
Notice the automatic 422 errors: we never wrote that validation code.

In [ ]:
from fastapi.testclient import TestClient
import importlib, main
importlib.reload(main)
client = TestClient(main.app)

print(client.get("/products", params={"max_price": 1000}).json())
print(client.get("/products/9").status_code, client.get("/products/9").json())
r = client.post("/products", json={"name": "Notebook", "price": 149}); print(r.status_code, r.json())
r = client.post("/products", json={"name": "Notebook", "price": 149}, headers={"X-API-Key": "demo-key"}); print(r.status_code, r.json())
r = client.post("/products", json={"name": "X", "price": -1}, headers={"X-API-Key": "demo-key"})
print(r.status_code, [(e["loc"][-1], e["msg"]) for e in r.json()["detail"]])

## Step 3 · The schema behind the docs
FastAPI builds an OpenAPI description of your API automatically.

In [ ]:
spec = client.get("/openapi.json").json()
for path, ops in spec["paths"].items():
    for method, op in ops.items():
        print(f"{method.upper():<5} {path:<18} {op.get('summary')}")

## Step 4 · Run it with Uvicorn and open the docs
Starts the server in the background. In Colab a window opens on the interactive docs at `/docs`.

In [ ]:
import subprocess, time, requests
api = subprocess.Popen(["uvicorn", "main:app", "--port", "8000"], stdout=open("uvicorn.log", "w"), stderr=subprocess.STDOUT)
time.sleep(4)
print(requests.get("http://127.0.0.1:8000/products").json())
def open_in_browser(port, path="/"):
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(port, path=path)
    except ImportError:
        print(f"Not in Colab: open http://localhost:{port}{path} in your browser")
open_in_browser(8000, "/docs")

## Step 5 · Stop the server
Clean up.

In [ ]:
api.terminate()
print("Uvicorn stopped")

## ✍️ Your turn
Using `client`, request `/products` with `in_stock=false` and store the **list of product names** returned in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == ["Desk Lamp"], f"Expected ['Desk Lamp'], got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = [p["name"] for p in client.get("/products", params={"in_stock": "false"}).json()]

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Path operation**
- **Pydantic model**
- **Validation**
- **OpenAPI docs**
- **async def**
- **Dependency**

**Next:** 40. Calling APIs (`requests · httpx · aiohttp`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/40_requests_httpx_aiohttp.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install fastapi[standard]
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.